# 📈 Bootstrap Uncertainty in Climatology Engine

This notebook introduces the Bootstrap method for estimating uncertainty of distribution parameters.

**What you will learn:**
- The concept of Bootstrap and its application in statistics
- Parametric Bootstrap method for uncertainty estimation
- Calculating Confidence Intervals
- Interpreting Bootstrap results
- Plotting uncertainty visualizations
- Comparing uncertainty across different models

---

## 📐 Bootstrap Theory

Bootstrap is a resampling method used to estimate the sampling distribution of a statistic.

### Parametric Bootstrap Method

1. Fit the distribution to the original data to estimate parameters $\hat{\theta}$
2. Generate $B$ random samples from the fitted distribution
3. Fit the distribution to each Bootstrap sample to obtain $\hat{\theta}^{(b)}$
4. Calculate confidence intervals from the Bootstrap distribution

### Percentile Confidence Interval

$$
CI_{95\%}(\theta) = [\theta_{(0.025)}, \theta_{(0.975)}]
$$

### Configurable Parameters

| Parameter | Default Value | Description |
|-----------|---------------|-------------|
| `n_bootstrap` | 100 | Number of Bootstrap iterations |
| `confidence_level` | 0.95 | Confidence level (95%) |
| `random_seed` | 42 | Random seed for reproducibility |

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from core.engine.plugin_loader import load_plugins
from core.uncertainty.bootstrap import bootstrap_fit

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11
print('✅ Libraries loaded.')

In [ ]:
# Load distribution plugins
plugins = load_plugins()
print(f'✅ Number of loaded distributions: {len(plugins)}')

for code, dist in plugins.items():
    print(f"   [{code}] {dist.name} (params: {dist.params})")

distributions = {dist.name: dist for dist in plugins.values()}

In [ ]:
# Load sample data
sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values

# Select tmean data for one year
data_year = data[:365, 1]

print(f'📊 Number of samples: {len(data_year)}')
print(f'   Mean: {np.mean(data_year):.2f}°C')
print(f'   Standard deviation: {np.std(data_year):.2f}°C')
print(f'   Sample size (n): {len(data_year)}')

In [ ]:
# Fit function for Bootstrap
def fit_func(data):
    """Fit function for Bootstrap"""
    dist = distributions['Normal']
    return dist.fit(data)

print("✅ Fit function defined.")

In [ ]:
# Run Bootstrap
n_bootstrap = 100  # For speed (use 100-1000 in practice)
print(f"\n🔄 Running Bootstrap with {n_bootstrap} iterations...")

cis = bootstrap_fit(data_year, fit_func, n_bootstrap=n_bootstrap, confidence=0.95)

print("\n📊 Bootstrap Results (95% CI):")
print("=" * 60)
for param, ci in cis.items():
    print(f"\n{param}:")
    print(f"   Estimated value: {ci['mean']:.4f}")
    print(f"   Confidence Interval: [{ci['lower']:.4f}, {ci['upper']:.4f}]")
    print(f"   Interval width: {ci['upper'] - ci['lower']:.4f}")
print("=" * 60)

In [ ]:
# Plot Bootstrap confidence intervals
fig, ax = plt.subplots(figsize=(10, 6))

params = list(cis.keys())
means = [cis[p]['mean'] for p in params]
lowers = [cis[p]['lower'] for p in params]
uppers = [cis[p]['upper'] for p in params]
errors = [means[i] - lowers[i] for i in range(len(params))]
errors_upper = [uppers[i] - means[i] for i in range(len(params))]

y_pos = np.arange(len(params))

ax.errorbar(means, y_pos, xerr=[errors, errors_upper],
            fmt='o', color='blue', capsize=8, capthick=2, 
            elinewidth=2, markersize=12, markeredgecolor='black')

ax.set_yticks(y_pos)
ax.set_yticklabels(params)
ax.set_xlabel('Parameter Value', fontsize=12)
ax.set_title('95% Confidence Intervals (Bootstrap)', fontsize=14, fontweight='bold')
ax.axvline(0, color='black', linestyle='-', alpha=0.2)
ax.grid(True, alpha=0.3, axis='x')

for i, (param, mean_val) in enumerate(zip(params, means)):
    ax.text(mean_val + 0.1, i, f'{mean_val:.3f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Bootstrap distribution for a specific parameter
def bootstrap_distribution_for_param(data, param_name, n_bootstrap=200):
    """
    Calculate Bootstrap distribution for a specific parameter
    """
    n = len(data)
    values = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=n, replace=True)
        res = fit_func(sample)
        if param_name in res and not np.isnan(res[param_name]):
            values.append(res[param_name])
    return np.array(values)

# Select a parameter (e.g., p1 = mean for Normal distribution)
param_to_plot = 'p1'
bootstrap_vals = bootstrap_distribution_for_param(data_year, param_to_plot, n_bootstrap=200)

print(f"📊 Bootstrap distribution for parameter '{param_to_plot}':")
print(f"   Number of samples: {len(bootstrap_vals)}")
print(f"   Mean: {np.mean(bootstrap_vals):.4f}")
print(f"   Standard deviation: {np.std(bootstrap_vals):.4f}")
print(f"   CI 95%: [{np.percentile(bootstrap_vals, 2.5):.4f}, {np.percentile(bootstrap_vals, 97.5):.4f}]")

In [ ]:
# Plot Bootstrap distribution histogram
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(bootstrap_vals, bins=30, density=True, alpha=0.6, 
        color='blue', edgecolor='black', label='Bootstrap Distribution')

# Plot confidence interval
lower = np.percentile(bootstrap_vals, 2.5)
upper = np.percentile(bootstrap_vals, 97.5)
mean_val = np.mean(bootstrap_vals)

ax.axvline(mean_val, color='red', linestyle='-', linewidth=2.5, label=f'Mean = {mean_val:.3f}')
ax.axvline(lower, color='green', linestyle='--', linewidth=2, label=f'CI 95%: [{lower:.3f}, {upper:.3f}]')
ax.axvline(upper, color='green', linestyle='--', linewidth=2)

ax.set_xlabel(f'Parameter {param_to_plot} Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title(f'Bootstrap Distribution for Parameter {param_to_plot} (Normal)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare uncertainty across different models
print("\n🔄 Calculating uncertainty for different models...")

uncertainty_results = []
for name, dist in distributions.items():
    try:
        def fit_func_dist(data):
            return dist.fit(data)
        
        cis_dist = bootstrap_fit(data_year, fit_func_dist, n_bootstrap=50, confidence=0.95)
        avg_width = np.mean([ci['upper'] - ci['lower'] for ci in cis_dist.values()])
        uncertainty_results.append({
            'Model': name,
            'Avg_Uncertainty': avg_width,
            'N_Params': len(cis_dist)
        })
        print(f"✅ {name}: Avg uncertainty = {avg_width:.4f}")
    except Exception as e:
        print(f"❌ {name}: Error - {str(e)}")

uncertainty_df = pd.DataFrame(uncertainty_results).sort_values('Avg_Uncertainty')
uncertainty_df

In [ ]:
# Plot uncertainty comparison
if not uncertainty_df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))

    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(uncertainty_df))]
    bars = ax.bar(uncertainty_df['Model'], uncertainty_df['Avg_Uncertainty'], 
                  color=colors, alpha=0.7, edgecolor='black', linewidth=1)

    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel('Average Uncertainty', fontsize=12)
    ax.set_title('Comparison of Uncertainty Across Models', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    for bar, val in zip(bars, uncertainty_df['Avg_Uncertainty']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()
else:
    print("❌ No data to plot.")

In [ ]:
# Effect of Bootstrap iterations on uncertainty
print("\n📊 Effect of Bootstrap iterations on uncertainty:")

n_iterations = [10, 20, 50, 100, 200]
avg_widths = []

for n in n_iterations:
    try:
        cis = bootstrap_fit(data_year, fit_func, n_bootstrap=n, confidence=0.95)
        avg_width = np.mean([ci['upper'] - ci['lower'] for ci in cis.values()])
        avg_widths.append(avg_width)
        print(f"   {n} iterations: Avg uncertainty = {avg_width:.4f}")
    except Exception as e:
        print(f"   {n} iterations: Error - {str(e)}")
        avg_widths.append(np.nan)

if len(avg_widths) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(n_iterations, avg_widths, 'bo-', linewidth=2, markersize=8, 
            markeredgecolor='black', markeredgewidth=1)
    ax.set_xlabel('Number of Bootstrap Iterations', fontsize=12)
    ax.set_ylabel('Average Uncertainty', fontsize=12)
    ax.set_title('Effect of Bootstrap Iterations on Uncertainty', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 📋 Summary

In this notebook you learned:

✅ The concept of Bootstrap and its application in uncertainty estimation
✅ Parametric Bootstrap method for confidence interval estimation
✅ Calculating 95% confidence intervals for distribution parameters
✅ Plotting uncertainty visualizations
✅ Comparing uncertainty across different models
✅ The effect of Bootstrap iterations on results

---

**Key Takeaways:**

1. Bootstrap is a non-parametric method for uncertainty estimation.
2. More Bootstrap iterations lead to more accurate results.
3. The 95% confidence interval represents the range containing 95% of estimates.
4. Models with lower uncertainty are more reliable.
5. For accurate results, at least 1000 Bootstrap iterations are recommended.

---

**Next Steps:**
- Notebook 07: Parallel Processing
- Notebook 08: Visualization
- Notebook 09: Custom Distribution